# IQ-235 — full source-checkpoint TPU inference
Select **Runtime → Change runtime type → TPU**. This runs the actual 235-layer model with BF16 expert streaming. The stored checkpoint uses the Mamba/Llama lexical foundation, not GLM-5.3 token IDs. No paid resources or training are launched.

In [ ]:
from pathlib import Path
import subprocess
repo = Path("/content/IQ")
if not repo.exists():
    subprocess.run(["git", "clone", "-b", "feat/iq235-colab-tpu-streaming",
                    "https://github.com/Harqer/IQ.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only",
                    "origin", "feat/iq235-colab-tpu-streaming"], check=True)
%cd /content/IQ
%pip install 'huggingface_hub==1.20.1' 'transformers==5.17.0' safetensors

Use Colab native TorchTPU and preserve the working TPU package versions. IQ text mode uses two public Llama 3.1 tokenizer distributions with a vocabulary and tokenization pipeline cross-check. Official or local tokenizer overrides are available.

In [ ]:
from huggingface_hub import login, HfApi, get_token
if not get_token():
    login(add_to_git_credential=False)
print("Signed in as:", HfApi().whoami(token=True)["name"])

## Optional: direct mounted checkpoint
If hf-mount is installed in the runtime, run the next cell before inference. It uses a read-only NFS mount by default. On Colab, mount permissions are not guaranteed. If the mount fails, the inference cell uses the bucket API. No full checkpoint download is required in either mode.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from huggingface_hub import get_token

def mount_environment():
    token = get_token()
    if not token:
        raise RuntimeError("Run the Hugging Face sign-in cell first")
    # hf-mount requires a bearer credential; use the active OAuth login
    # for this child process only, never print it or place it in arguments.
    return dict(os.environ, HF_TOKEN=token)
MOUNT = Path("/content/iq-weights")
RUN = MOUNT / "IQ/runs/glm53-full78-local-20261025a"
if shutil.which("hf-mount"):
    MOUNT.mkdir(exist_ok=True)
    if not (RUN / "iq_config.json").is_file():
        subprocess.run(["hf-mount", "start", "--read-only", "--cache-size",
                        "10000000000", "bucket", "kofecheckmate/iq-glm53-all256",
                        str(MOUNT)], env=mount_environment(), check=False)
    print("Mount visible:", (RUN / "iq_config.json").is_file())
else:
    print("hf-mount not installed; see docs/IQ235_COLAB_TPU.md for installation.")

In [ ]:
%cd /content/IQ
from pathlib import Path
from scripts.iq235_colab_tpu import open_session

# Initialize once; reuse the same session in the question cell below.
if "iq" not in globals() or iq._closed:
    mount = Path("/content/iq-weights")
    run = mount / "IQ/runs/glm53-full78-local-20261025a"
    iq = open_session(mount_dir=mount if (run / "iq_config.json").is_file() else None)

In [ ]:
QUESTION = "What is the capital of France?"
result = iq.ask(QUESTION, max_new_tokens=4)

Rerun only the question cell for each independent question. The model, tokenizer, and cache stay open. Close with iq.close() when finished. A Colab runtime reset discards temporary disk. This is base-model completion, not proven instruction tuning.